# Judge-context sensitivity run (summarization)
In the main run the self-evaluation (judge) prompt saw only the first 1,500 characters of the article, whereas the generator saw up to 3,000. This notebook repeats the tau = 7 loop for summarization with the judge shown 1,500 and 3,000 characters, everything else identical to the main run (models, quantization, prompts, greedy decoding, parser, scoring).

- 9 models x first 50 summarization samples (sample_idx 0-49 of the main run).
- The first-turn summary is generated once and shared by both conditions (greedy decoding makes it identical), so the conditions differ only in what the judge sees and in the retries that follow.
- Output: `agentic_energy_results_v4/judge_context_run/judge_context.csv` (one row per model, sample and condition) and `judge_context_summary.csv`. The run is resume-safe.
- Energy is not tracked; wall-clock time of the loop after the first turn is recorded instead.


## 1. GPU check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install

In [ ]:
!pip install -q transformers accelerate bitsandbytes codecarbon datasets evaluate rouge_score sentencepiece
!pip uninstall -y -q hf_xet 2>/dev/null || true

## 3. Drive + cache setup (run BEFORE importing transformers)

In [ ]:
import sys, os, tempfile
sys.modules['hf_xet'] = None
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE_DIR = "/content/drive/MyDrive/hf_cache"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_CACHE_DIR
os.environ["HF_HUB_DISABLE_XET"] = "1"
DRIVE_TMPDIR = os.path.join(DRIVE_CACHE_DIR, "tmp"); os.makedirs(DRIVE_TMPDIR, exist_ok=True)
for k in ("TMPDIR", "TEMP", "TMP"): os.environ[k] = DRIVE_TMPDIR
tempfile.tempdir = DRIVE_TMPDIR

RESULTS_DIR = "/content/drive/MyDrive/agentic_energy_results_v4/judge_context_run"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results dir:", RESULTS_DIR)
if "huggingface_hub" in sys.modules:
    print("WARNING: huggingface_hub already imported; restart runtime and run this cell first.")

## 3b. Clean stale/partial downloads and check disk (run after every reconnect)

In [ ]:
import glob, subprocess
hub_dir = os.path.join(os.environ["HF_HOME"], "hub")
removed = 0
for f in glob.glob(os.path.join(hub_dir, "**", "*.incomplete"), recursive=True):
    print("removing partial download:", f); os.remove(f); removed += 1
for sub in ("tmp", "offload"):
    d = os.path.join(DRIVE_CACHE_DIR, sub)
    for f in glob.glob(os.path.join(d, "**", "*"), recursive=True):
        if os.path.isfile(f):
            os.remove(f); removed += 1
print(f"Removed {removed} stale files.")
print("Local disk:", subprocess.run(["df", "-h", "/"], capture_output=True, text=True).stdout.splitlines()[-1])
print("Drive hf_cache size:", subprocess.run(["du", "-sh", DRIVE_CACHE_DIR], capture_output=True, text=True).stdout.strip())
print("Drive free:", subprocess.run(["df", "-h", "/content/drive/MyDrive"], capture_output=True, text=True).stdout.splitlines()[-1])

## 4. Hugging Face login (token from Colab Secrets: add a secret named HF_TOKEN via the key icon in the left sidebar)

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(userdata.get("HF_TOKEN"))
print("Logged in as:", whoami()["name"])

## 5. Imports, environment record

In [6]:
import re, time, json, gc, shutil, platform
import pandas as pd
from datetime import datetime, timezone
import transformers, accelerate, bitsandbytes, codecarbon, datasets, evaluate
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig
from codecarbon import EmissionsTracker

env_info = {
    "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__, "codecarbon": codecarbon.__version__,
    "datasets": datasets.__version__, "evaluate": evaluate.__version__,
    "gpu": torch.cuda.get_device_name(0), "recorded_at": datetime.now(timezone.utc).isoformat(),
}
with open(os.path.join(RESULTS_DIR, "env_info.json"), "w") as f: json.dump(env_info, f, indent=2)
print(json.dumps(env_info, indent=2))

{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "transformers": "5.21.0",
  "accelerate": "1.15.0",
  "bitsandbytes": "0.50.2",
  "codecarbon": "3.3.2",
  "datasets": "4.8.5",
  "evaluate": "0.4.6",
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "recorded_at": "2026-11-02T23:18:06.867941+00:00"
}


## 6. Configuration

In [ ]:
MODEL_REGISTRY = {
    "llama-3.2-3b":  {"hf_id": "meta-llama/Llama-3.2-3B-Instruct",            "load_in_4bit": True, "category": "small"},
    "qwen2.5-3b":    {"hf_id": "Qwen/Qwen2.5-3B-Instruct",                     "load_in_4bit": True, "category": "small"},
    "gemma-2-2b":    {"hf_id": "google/gemma-2-2b-it",                         "load_in_4bit": True, "category": "small"},
    "llama-3-8b":    {"hf_id": "meta-llama/Meta-Llama-3-8B-Instruct",          "load_in_4bit": True, "category": "medium"},
    "qwen2.5-7b":    {"hf_id": "Qwen/Qwen2.5-7B-Instruct",                     "load_in_4bit": True, "category": "medium"},
    "mistral-7b":    {"hf_id": "mistralai/Mistral-7B-Instruct-v0.3",           "load_in_4bit": True, "category": "medium"},
    "llama-3-70b":   {"hf_id": "unsloth/llama-3-70b-Instruct-bnb-4bit",        "prequantized": True, "category": "large"},
    "qwen2.5-72b":   {"hf_id": "unsloth/Qwen2.5-72B-Instruct-bnb-4bit",        "prequantized": True, "category": "large"},
    "mistral-large": {"hf_id": "unsloth/Mistral-Large-Instruct-2407-bnb-4bit", "prequantized": True, "category": "large"},
}

MAX_EXTRA_TURNS = 2
THRESHOLDS = [7, 5, 9]           # main run first
JUDGE_MAX_NEW_TOKENS = 8
TASK_MAX_TOKENS = {"summarization": 128, "classification": 20, "math": 512}
ACTIVE_TASKS = ["summarization"]
N_SAMPLES_PER_TASK = 50
SEED = 42
FLUSH_EVERY = 10

def csv_path_for(tau):
    return os.path.join(RESULTS_DIR, f"agentic_energy_v4_threshold{tau}.csv")

## 7. Model loading (records device map and quantization config)

In [ ]:
MODEL_INFO_PATH = os.path.join(RESULTS_DIR, "model_info.json")

def load_model(model_key):
    cfg = MODEL_REGISTRY[model_key]
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_id"], cache_dir=DRIVE_CACHE_DIR)
    gc.collect(); torch.cuda.empty_cache()
    free_gib = torch.cuda.mem_get_info()[0] / 1024**3
    safe_gib = max(free_gib - 4, 8)
    print(f"  [{model_key}] free GPU memory {free_gib:.1f} GiB, limit {safe_gib:.0f} GiB")
    load_kwargs = {"device_map": "auto", "max_memory": {0: f"{safe_gib:.0f}GiB"},
                   "cache_dir": DRIVE_CACHE_DIR, "low_cpu_mem_usage": True}
    if cfg.get("prequantized"):
        model_config = AutoConfig.from_pretrained(cfg["hf_id"], cache_dir=DRIVE_CACHE_DIR)
        load_kwargs["config"] = model_config
    else:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(cfg["hf_id"], **load_kwargs)

    # --- record placement + quantization config ---
    devmap = getattr(model, "hf_device_map", {}) or {}
    devices = sorted({str(v) for v in devmap.values()})
    qc = getattr(model.config, "quantization_config", None)
    if qc is not None and not isinstance(qc, dict):
        qc = qc.to_dict() if hasattr(qc, "to_dict") else vars(qc)
    n_params = sum(p.numel() for p in model.parameters())
    info = {"hf_id": cfg["hf_id"], "category": cfg["category"], "devices_used": devices,
            "n_layers_offloaded": sum(1 for v in devmap.values() if str(v) not in ("0", "cuda:0")),
            "quantization_config": {k: str(v) for k, v in (qc or {}).items()},
            "param_count_reported": n_params, "loaded_at": datetime.now(timezone.utc).isoformat()}
    all_info = json.load(open(MODEL_INFO_PATH)) if os.path.exists(MODEL_INFO_PATH) else {}
    all_info[model_key] = info
    json.dump(all_info, open(MODEL_INFO_PATH, "w"), indent=2)
    print(f"  [{model_key}] devices={devices} offloaded_layers={info['n_layers_offloaded']} "
          f"compute_dtype={info['quantization_config'].get('bnb_4bit_compute_dtype')}")
    assert info["n_layers_offloaded"] == 0, "Layers were offloaded to CPU/disk; energy would be inflated. Stop."
    return model, tokenizer


def format_chat_prompt(tokenizer, user_content):
    try:
        return tokenizer.apply_chat_template([{"role": "user", "content": user_content}],
                                             tokenize=False, add_generation_prompt=True)
    except Exception:
        return user_content


def generate_text(model, tokenizer, prompt, max_new_tokens):
    prompt = format_chat_prompt(tokenizer, prompt)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, pad_token_id=tokenizer.eos_token_id)
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return text, int(new_ids.shape[0])

## 8. Task data

In [ ]:
from datasets import load_dataset

def _extract_gsm8k_answer(answer_field):
    m = re.search(r"####\s*([\-0-9,\.]+)", answer_field)
    return m.group(1).replace(",", "").strip() if m else None

_DATA_CACHE = {}
def load_task_data(task_name, n_samples=N_SAMPLES_PER_TASK, seed=SEED):
    if task_name in _DATA_CACHE: return _DATA_CACHE[task_name]
    if task_name == "summarization":
        ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test", cache_dir=DRIVE_CACHE_DIR)
        ds = ds.shuffle(seed=seed).select(range(n_samples))
        data = [{"input": ex["article"], "reference": ex["highlights"]} for ex in ds]
    elif task_name == "classification":
        ds = load_dataset("fancyzhx/ag_news", split="test", cache_dir=DRIVE_CACHE_DIR)
        ds = ds.shuffle(seed=seed).select(range(n_samples))
        names = ds.features["label"].names
        data = [{"input": ex["text"], "reference": names[ex["label"]]} for ex in ds]
    elif task_name == "math":
        ds = load_dataset("openai/gsm8k", "main", split="test", cache_dir=DRIVE_CACHE_DIR)
        ds = ds.shuffle(seed=seed).select(range(n_samples))
        data = [{"input": ex["question"], "reference": _extract_gsm8k_answer(ex["answer"])} for ex in ds]
    else:
        raise ValueError(task_name)
    _DATA_CACHE[task_name] = data
    return data

## 9. Prompts and self-confidence scoring

In [ ]:
def build_base_prompt(task_name, model_input):
    if task_name == "summarization":
        return ("Summarize the following news article in 2-3 sentences. "
                "Write the summary in English.\n\n" + model_input[:3000])
    elif task_name == "classification":
        return ("Classify the following news text into exactly ONE of these four categories: "
                "World, Sports, Business, Sci/Tech.\n\n"
                "Respond with ONLY the category name (World, Sports, Business or Sci/Tech). "
                "Do not add any explanation or other words.\n\n"
                f"Text: {model_input[:2000]}\n\nCategory:")
    elif task_name == "math":
        return ("Solve the following math problem step by step. "
                "On the last line, write your final answer in the format 'Answer: <number>'.\n\n"
                f"Problem: {model_input}")
    raise ValueError(task_name)


def build_retry_prompt(base_prompt, previous_output, confidence):
    return (f"{base_prompt}\n\n"
            f"Your previous answer was: {previous_output}\n"
            f"That answer received a low confidence score ({confidence}/10). "
            "Produce a more accurate and clearer answer:")


SCORE_RE = re.compile(r"\b(10|[1-9])\b")

def parse_confidence(raw):
    m = SCORE_RE.search(raw)
    if m: return int(m.group(1)), False
    return 5, True          # default on failure, flagged


def self_confidence_score(model, tokenizer, task_name, model_input, model_output):
    judge_prompt = (
        f"Below is an output produced for a {task_name} task.\n\n"
        f"INPUT: {model_input[:1500]}\n\n"
        f"OUTPUT: {model_output}\n\n"
        "Rate the correctness and quality of this output on a scale from 1 (very poor) to 10 (perfect). "
        "Reply with ONLY a single number. Do not write anything else.")
    raw, _ = generate_text(model, tokenizer, judge_prompt, JUDGE_MAX_NEW_TOKENS)
    score, failed = parse_confidence(raw)
    return max(1, min(score, 10)), raw, failed

## 10. Agentic loop (as in notebook 01; this run uses the variant defined in section 13)

In [ ]:
def agentic_loop(model, tokenizer, task_name, sample, threshold):
    base_prompt = build_base_prompt(task_name, sample["input"])
    prompt = base_prompt
    max_toks = TASK_MAX_TOKENS[task_name]
    conf_hist, outputs, judge_raw, ntoks, failed_flags = [], [], [], [], []
    for turn in range(MAX_EXTRA_TURNS + 1):
        output, n_new = generate_text(model, tokenizer, prompt, max_toks)
        conf, raw, failed = self_confidence_score(model, tokenizer, task_name, sample["input"], output)
        outputs.append(output); ntoks.append(n_new); conf_hist.append(conf)
        judge_raw.append(raw); failed_flags.append(failed)
        if conf >= threshold or turn == MAX_EXTRA_TURNS:
            break
        prompt = build_retry_prompt(base_prompt, output, conf)
    return {"final_output": outputs[-1], "n_turns": len(outputs), "confidence_history": conf_hist,
            "per_turn_outputs": outputs, "judge_raw_history": judge_raw,
            "n_new_tokens_history": ntoks, "truncated_any": any(n >= max_toks for n in ntoks),
            "parse_failed_any": any(failed_flags), "n_parse_failed": sum(failed_flags)}

## 11. Energy measurement (as in notebook 01; not used in this run)

In [ ]:
def run_with_energy_tracking(model, tokenizer, task_name, sample, threshold, run_id):
    tracker = EmissionsTracker(project_name=run_id, output_dir=RESULTS_DIR, log_level="error",
                               save_to_file=False, measure_power_secs=0.1)
    tracker.start(); t0 = time.time()
    res = agentic_loop(model, tokenizer, task_name, sample, threshold)
    elapsed = time.time() - t0
    emissions = tracker.stop()
    def _kwh(attr):
        v = getattr(tracker, attr, None)
        return getattr(v, "kWh", None) if v is not None else None
    res.update({"elapsed_sec": elapsed, "emissions_kg_co2eq": emissions,
                "energy_kwh": _kwh("_total_energy"), "gpu_energy_kwh": _kwh("_total_gpu_energy"),
                "cpu_energy_kwh": _kwh("_total_cpu_energy"), "ram_energy_kwh": _kwh("_total_ram_energy")})
    return res

## 12. Scoring

In [ ]:
rouge = evaluate.load("rouge")

def score_task(task_name, prediction, reference):
    if task_name == "summarization":
        return {"rougeL": rouge.compute(predictions=[prediction], references=[reference])["rougeL"]}
    if task_name == "classification":
        return {"correct": int(reference.strip().lower() in prediction.strip().lower())}
    if task_name == "math":
        nums = re.findall(r"[\-]?\d[\d,]*\.?\d*", prediction)
        if not nums or reference is None: return {"correct": 0}
        try:
            return {"correct": int(abs(float(nums[-1].replace(",", "")) - float(reference)) < 1e-4)}
        except ValueError:
            return {"correct": 0}
    raise ValueError(task_name)

## 13. Run both judge-context conditions

In [ ]:
JUDGE_CHARS = (1500, 3000)          # original limit and the full article length seen by the generator
TAU = 7
JUDGE_CSV = os.path.join(RESULTS_DIR, "judge_context.csv")


def judge_score(model, tokenizer, task_name, model_input, model_output, chars):
    """Same judge prompt as the main run; only the input truncation length differs."""
    prompt = (
        f"Below is an output produced for a {task_name} task.\n\n"
        f"INPUT: {model_input[:chars]}\n\n"
        f"OUTPUT: {model_output}\n\n"
        "Rate the correctness and quality of this output on a scale from 1 (very poor) to 10 (perfect). "
        "Reply with ONLY a single number. Do not write anything else.")
    raw, _ = generate_text(model, tokenizer, prompt, JUDGE_MAX_NEW_TOKENS)
    score, failed = parse_confidence(raw)
    return max(1, min(score, 10)), raw, failed


def timed(fn, *args):
    t0 = time.time(); out = fn(*args); return out, time.time() - t0


def loop_from_first(model, tokenizer, sample, first_output, chars):
    """Run the tau = 7 loop for one judge-context condition, reusing the shared first-turn output."""
    task = "summarization"
    base_prompt = build_base_prompt(task, sample["input"])
    outputs, confs, raws, secs = [first_output], [], [], 0.0
    for turn in range(MAX_EXTRA_TURNS + 1):
        (conf, raw, failed), dt = timed(judge_score, model, tokenizer, task, sample["input"], outputs[-1], chars)
        confs.append(conf); raws.append(raw); secs += dt
        if conf >= TAU or turn == MAX_EXTRA_TURNS:
            break
        (out, _), dt = timed(generate_text, model, tokenizer,
                             build_retry_prompt(base_prompt, outputs[-1], conf), TASK_MAX_TOKENS[task])
        outputs.append(out); secs += dt
    return outputs, confs, raws, secs


def run_judge_context(model_keys=list(MODEL_REGISTRY.keys())):
    rows = pd.read_csv(JUDGE_CSV).to_dict("records") if os.path.exists(JUDGE_CSV) else []
    done = {(r["model"], r["sample_idx"]) for r in rows}
    data = load_task_data("summarization")
    for model_key in model_keys:
        remaining = [i for i in range(N_SAMPLES_PER_TASK) if (model_key, i) not in done]
        if not remaining:
            print(f"[{model_key}] complete, skipping."); continue
        print(f"\n=== Loading {model_key} ===")
        model, tokenizer = load_model(model_key)
        for count, idx in enumerate(remaining, 1):
            sample = data[idx]
            (first, _), gen_s = timed(generate_text, model, tokenizer,
                                      build_base_prompt("summarization", sample["input"]), TASK_MAX_TOKENS["summarization"])
            first_rouge = score_task("summarization", first, sample["reference"])["rougeL"]
            for chars in JUDGE_CHARS:
                outs, confs, raws, secs = loop_from_first(model, tokenizer, sample, first, chars)
                rows.append({
                    "model": model_key, "model_category": MODEL_REGISTRY[model_key]["category"],
                    "sample_idx": idx, "judge_chars": chars, "article_chars": len(sample["input"]),
                    "n_turns": len(outs), "confidence_history": json.dumps(confs),
                    "judge_raw_history": json.dumps(raws, ensure_ascii=False),
                    "first_rougeL": first_rouge,
                    "final_rougeL": score_task("summarization", outs[-1], sample["reference"])["rougeL"],
                    "final_output": outs[-1], "first_gen_sec": gen_s, "loop_sec_after_first": secs,
                    "timestamp": datetime.now(timezone.utc).isoformat()})
            done.add((model_key, idx))
            if count % FLUSH_EVERY == 0 or count == len(remaining):
                pd.DataFrame(rows).to_csv(JUDGE_CSV, index=False)
                print(f"     [{count}/{len(remaining)}] saved")
        del model, tokenizer
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        if MODEL_REGISTRY[model_key]["category"] == "large":
            cache_path = os.path.join(os.environ["HF_HOME"], "hub", "models--" + MODEL_REGISTRY[model_key]["hf_id"].replace("/", "--"))
            shutil.rmtree(cache_path, ignore_errors=True)
    print("\nJudge-context run complete:", JUDGE_CSV)

run_judge_context()


## 14. Compare the two conditions

In [15]:
import numpy as np
df = pd.read_csv(JUDGE_CSV)
df["first_conf"] = df["confidence_history"].apply(lambda s: json.loads(s)[0])
df["retried"] = (df["n_turns"] > 1).astype(int)
df["gain"] = df["final_rougeL"] - df["first_rougeL"]
w = df.pivot_table(index=["model", "model_category", "sample_idx"], columns="judge_chars",
                   values=["first_conf", "retried", "n_turns", "final_rougeL", "gain", "loop_sec_after_first"]).reset_index()
rng = np.random.default_rng(0)

def boot_ci(x, n=2000):
    x = np.asarray(x, float)
    b = [rng.choice(x, len(x)).mean() for _ in range(n)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

rows = []
for (m, cat), g in w.groupby(["model", "model_category"]):
    dconf = g[("first_conf", 3000)] - g[("first_conf", 1500)]
    dfinal = g[("final_rougeL", 3000)] - g[("final_rougeL", 1500)]
    lo, hi = boot_ci(dfinal)
    rows.append({
        "model": m, "category": cat, "n": len(g),
        "conf_1500": g[("first_conf", 1500)].mean(), "conf_3000": g[("first_conf", 3000)].mean(),
        "mean_conf_change": dconf.mean(), "share_score_changed": (dconf != 0).mean(),
        "retry_rate_1500": g[("retried", 1500)].mean(), "retry_rate_3000": g[("retried", 3000)].mean(),
        "decision_change_rate": (g[("retried", 1500)] != g[("retried", 3000)]).mean(),
        "turns_1500": g[("n_turns", 1500)].mean(), "turns_3000": g[("n_turns", 3000)].mean(),
        "gain_1500": g[("gain", 1500)].mean(), "gain_3000": g[("gain", 3000)].mean(),
        "final_rougeL_1500": g[("final_rougeL", 1500)].mean(), "final_rougeL_3000": g[("final_rougeL", 3000)].mean(),
        "final_diff_3000_minus_1500": dfinal.mean(), "final_diff_ci_lo": lo, "final_diff_ci_hi": hi,
        "loop_sec_1500": g[("loop_sec_after_first", 1500)].mean(), "loop_sec_3000": g[("loop_sec_after_first", 3000)].mean()})
summary = pd.DataFrame(rows).round(4)
summary.to_csv(os.path.join(RESULTS_DIR, "judge_context_summary.csv"), index=False)
pd.set_option("display.width", 250)
print(summary.to_string(index=False))
print("\nShare of articles longer than 1,500 characters:", (df.drop_duplicates("sample_idx")["article_chars"] > 1500).mean().round(3))
print("Pooled decision change rate:", (w[("retried", 1500)] != w[("retried", 3000)]).mean().round(3))
print("Pooled mean first-turn confidence change:", (w[("first_conf", 3000)] - w[("first_conf", 1500)]).mean().round(3))


        model category  n  conf_1500  conf_3000  mean_conf_change  share_score_changed  retry_rate_1500  retry_rate_3000  decision_change_rate  turns_1500  turns_3000  gain_1500  gain_3000  final_rougeL_1500  final_rougeL_3000  final_diff_3000_minus_1500  final_diff_ci_lo  final_diff_ci_hi  loop_sec_1500  loop_sec_3000
   gemma-2-2b    small 50       8.44       8.32             -0.12                 0.20             0.00             0.00                  0.00        1.00        1.00     0.0000     0.0000             0.2187             0.2187                      0.0000            0.0000            0.0000         0.0817         0.0860
  llama-3-70b    large 50       8.76       8.98              0.22                 0.30             0.00             0.00                  0.00        1.00        1.00     0.0000     0.0000             0.2010             0.2010                      0.0000            0.0000            0.0000         0.3869         0.4860
   llama-3-8b   medium 50       7.80 